In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [43]:
# ==========================================
# 1. ĐỌC VÀ LÀM SẠCH DỮ LIỆU CƠ BẢN
# ==========================================
missing_values = ["NaN "]
df = pd.read_csv("../dataset/foodDeli/train.csv", na_values=missing_values)
test_df = pd.read_csv("../dataset/foodDeli/test.csv", na_values=missing_values)
submission_df = pd.read_csv("../dataset/foodDeli/Sample_Submission.csv")

In [44]:

# Xử lý tập Test & Submission
test_df['ID'] = test_df['ID'].astype(str).str.strip()
submission_df['ID'] = submission_df['ID'].astype(str).str.strip()
test_df.dropna(inplace=True)
test_df['Weather_conditions'] = test_df['Weather_conditions'].str.replace('conditions ', '', regex=True)
test_df = pd.merge(test_df, submission_df, on='ID', how='left')

# Xử lý tập Train
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
df.columns = df.columns.str.strip()
df.dropna(inplace=True)
df['Weather_conditions'] = df['Weather_conditions'].str.replace('conditions ', '', regex=True)

# Xử lý biến mục tiêu (Time_taken) trên tập Train
df.rename(columns={'Time_taken(min)': 'Time_taken'}, inplace=True)
df['Time_taken'] = df['Time_taken'].astype(str).str.replace('(min)', '', regex=False).str.strip()
df['Time_taken'] = pd.to_numeric(df['Time_taken'])

In [45]:
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1)/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2.0)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

# Áp dụng cho Train
df['Distance_km'] = haversine_distance(
    df['Restaurant_latitude'], df['Restaurant_longitude'],
    df['Delivery_location_latitude'], df['Delivery_location_longitude'])

df['Order_Time_Full'] = pd.to_datetime(df['Order_Date'] + ' ' + df['Time_Orderd'])
df['Picked_Time_Full'] = pd.to_datetime(df['Order_Date'] + ' ' + df['Time_Order_picked'])
df['Preparation_Time'] = (df['Picked_Time_Full'] - df['Order_Time_Full']).dt.total_seconds() / 60.0
df['Preparation_Time'] = np.where(df['Preparation_Time'] < 0, df['Preparation_Time'] + 1440, df['Preparation_Time'])

# Áp dụng cho Test
test_df['Distance_km'] = haversine_distance(
    test_df['Restaurant_latitude'], test_df['Restaurant_longitude'],
    test_df['Delivery_location_latitude'], test_df['Delivery_location_longitude'])

test_df['Order_Time_Full'] = pd.to_datetime(test_df['Order_Date'] + ' ' + test_df['Time_Orderd'])
test_df['Picked_Time_Full'] = pd.to_datetime(test_df['Order_Date'] + ' ' + test_df['Time_Order_picked'])
test_df['Preparation_Time'] = (test_df['Picked_Time_Full'] - test_df['Order_Time_Full']).dt.total_seconds() / 60.0
test_df['Preparation_Time'] = np.where(test_df['Preparation_Time'] < 0, test_df['Preparation_Time'] + 1440, test_df['Preparation_Time'])

In [46]:
# ==========================================
# 3. MÃ HÓA ONE-HOT VÀ XÓA CỘT THÔ
# ==========================================
categorical_cols = ['Weather_conditions', 'Road_traffic_density', 'Type_of_order', 'Type_of_vehicle','Vehicle_condition', 'Festival', 'City']

# One-hot
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Chuyển boolean sang int
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

for col in test_df.columns:
    if test_df[col].dtype == 'bool':
        test_df[col] = test_df[col].astype(int)

# Xóa các cột định danh và cột thô đã được trích xuất
cols_to_drop = [
    'Delivery_person_ID', 'multiple_deliveries', 
    'Restaurant_latitude', 'Restaurant_longitude',
    'Delivery_location_latitude', 'Delivery_location_longitude',
    'Order_Date', 'Time_Orderd', 'Time_Order_picked',
    'Order_Time_Full', 'Picked_Time_Full'
]

df.drop(columns=cols_to_drop, errors='ignore', inplace=True)
test_df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

In [47]:
# ==========================================
# 4. TÁCH BIẾN VÀ CHIA TẬP 
# ==========================================
y = df['Time_taken']
X = df.drop(columns=['Time_taken'])

# Tách Train/Val ngay lúc này!
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

y_test = test_df['Time_taken']
X_test = test_df.drop(columns=['Time_taken'])

# Ép khuôn cột cho Val và Test theo đúng Train
X_train_cols = X_train.columns
X_val = X_val.reindex(columns=X_train_cols, fill_value=0)
X_test = X_test.reindex(columns=X_train_cols, fill_value=0)


In [48]:
# ==========================================
# 5. XỬ LÝ NGOẠI LAI (CHỈ TÁC ĐỘNG LÊN TẬP TRAIN)
# ==========================================
Q1_dist = X_train['Distance_km'].quantile(0.25)
Q3_dist = X_train['Distance_km'].quantile(0.75)
lower_dist = Q1_dist - 1.5 * (Q3_dist - Q1_dist)
upper_dist = Q3_dist + 1.5 * (Q3_dist - Q1_dist)

# Tìm Index của ngoại lai và Drop khỏi X_train, y_train
outlier_idx = X_train[(X_train['Distance_km'] < lower_dist) | (X_train['Distance_km'] > upper_dist)].index

X_train = X_train.drop(outlier_idx)
y_train = y_train.drop(outlier_idx)

print(f"Kích thước TRAIN sau khi dọn ngoại lai: {X_train.shape} (Đã xóa {len(outlier_idx)} dòng)")

Kích thước TRAIN sau khi dọn ngoại lai: (32987, 23) (Đã xóa 107 dòng)


In [49]:
# ==========================================
# 6. CHUẨN HÓA DỮ LIỆU (STANDARD SCALER)
# ==========================================
scaler = StandardScaler()
cols_to_scale = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Distance_km', 'Preparation_Time']

# Fit trên Train, Transform trên Train, Val, Test
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_val[cols_to_scale] = scaler.transform(X_val[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [50]:
# ==========================================
# 7. GỘP BIẾN VÀ LƯU FILE CUỐI CÙNG
# ==========================================
train_data_final = pd.concat([X_train, y_train], axis=1)
val_data_final = pd.concat([X_val, y_val], axis=1)
test_data_final = pd.concat([X_test, y_test], axis=1)

train_file = "../dataset/foodDeli_processed/train_processed.csv"
val_file = "../dataset/foodDeli_processed/val_processed.csv"
test_file = "../dataset/foodDeli_processed/test_processed.csv"

train_data_final.to_csv(train_file, index=False, encoding='utf-8-sig')
val_data_final.to_csv(val_file, index=False, encoding='utf-8-sig')
test_data_final.to_csv(test_file, index=False, encoding='utf-8-sig')